# Índice de Desigualdade Socioespacial por eixo — Curitiba e Porto Alegre

Um painel único com os quatro indicadores (IDS e seus três eixos) para as duas cidades. Serve para ver, de relance, onde a vulnerabilidade se concentra e se os eixos concordam ou divergem espacialmente.

O IDS é a média de três eixos normalizados por min–max **dentro de cada município**: esgotamento inadequado, renda (invertida) e taxa de analfabetismo. Cada eixo vai de 0 a 1, onde 1 indica maior vulnerabilidade.

**Importante para a leitura:** como a normalização é feita por município, o valor 0,5 em Curitiba não significa o mesmo que 0,5 em Porto Alegre. A comparação válida entre cidades é de **padrão espacial**, não de magnitude.


## 1. Bibliotecas e caminhos


In [26]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import colors as mpl_colors

BASE_DIR = Path.cwd().parent.parent.parent
RESULTS_DIR = BASE_DIR / "results"
OUTPUT_DIR = Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CIDADES = {
    "curitiba": "Curitiba",
    "porto_alegre": "Porto Alegre",
}

print("raiz:", BASE_DIR)
print("resultados:", RESULTS_DIR)


raiz: /home/matheus/Documentos/mestrado/dissertacao
resultados: /home/matheus/Documentos/mestrado/dissertacao/results


## 2. Carregamento da malha e dos índices

A malha é lida de `prepared/grid.parquet` e os índices de `prepared/ids.parquet`, gerados por `scripts/analysis/compute_ids.py`. Os dois arquivos são unidos por `cell_idx`.


In [37]:
malhas = {}
for cidade in CIDADES:
    grid = gpd.read_parquet(RESULTS_DIR / cidade / "prepared" / "grid.parquet")
    ids = pd.read_parquet(RESULTS_DIR / cidade / "prepared" / "ids.parquet")
    malhas[cidade] = grid.merge(ids, on="cell_idx", how="left")
    d = malhas[cidade]
    print(f"{cidade:14s} células={len(d):5d}  IDS com valor={d['ids'].notna().sum():5d}  "
          f"população={d['populacao'].sum():10.0f}")


curitiba       células= 4398  IDS com valor= 4355  população=   1711767
porto_alegre   células= 5065  IDS com valor= 4729  população=   1276719


## 3. Colunas e escala de cores

Os quatro indicadores são plotados com a **mesma escala fixa de 0 a 1**, porque todos já estão normalizados na mesma unidade. Usar uma escala compartilhada é o que permite comparar os eixos entre si — se cada um tivesse a própria faixa, um eixo com pouca variação pareceria tão contrastado quanto um com muita.

A paleta é `magma`, escolhida por ser sequencial perceptualmente ordenada: o olho lê a progressão sem precisar consultar a legenda, e ela mantém contraste mesmo nas extremidades escuras.


In [38]:
EIXOS = [
    ("ids", "IDS (composto)"),
    ("eixo_sem_esgoto", "Esgoto inadequado"),
    ("eixo_renda", "Renda (invertida)"),
    ("eixo_analfabetismo", "Taxa de analfabetismo"),
]

CMAP = "magma"
V_MIN, V_MAX = 0.0, 1.0
NORM = mpl_colors.Normalize(vmin=V_MIN, vmax=V_MAX, clip=True)

# ---------------------------------------------------------------------------
# Sem mascaramento.
#
# Cada eixo já carrega o próprio NaN: os quatro têm exatamente a mesma
# contagem de ausentes (43 células em Curitiba, 336 em Porto Alegre), e são as
# células que não intersectam nenhum setor censitário. O geopandas pinta essas
# células de cinza pelo ``missing_kwds``, sem precisar de intervenção.
#
# Havia antes uma máscara que apagava os eixos onde a POPULAÇÃO era nula. Isso
# era errado: a população é interpolada por densidade × área e falha em 221
# células de Curitiba e 680 de Porto Alegre, das quais 178 e 344 têm os três
# eixos perfeitamente calculados. A máscara escondia dado válido — o eixo de
# renda, por exemplo, aparecia com 221 células cinzas quando só tinha 43. A
# população só é necessária mais tarde, no F15, e não entra em nenhum destes
# quatro mapas.
# ---------------------------------------------------------------------------
COR_SEM_DADO = "#bdbdbd"

# ---------------------------------------------------------------------------
# Diagnóstico: média e desvio de cada eixo. Duas coisas a observar: (a) eixos
# com desvio baixo vão aparecer quase uniformes no mapa, por mais que a escala
# seja 0–1; (b) a média não é comparável entre cidades, apenas o padrão
# espacial.
# ---------------------------------------------------------------------------
for cidade in CIDADES:
    d = malhas[cidade]
    contagens = "  ".join(
        f"{rotulo.split('(')[0].strip()}: {int(d[col].isna().sum())}"
        for col, rotulo in EIXOS
    )
    print(f"{cidade:14s} células={len(d):5d}  NaN por eixo -> {contagens}")

print()
print(f"{'cidade':>14}  {'eixo':>24}  {'média':>6}  {'desvio':>7}  {'p95':>6}")
for cidade in CIDADES:
    for coluna, rotulo in EIXOS:
        s = malhas[cidade][coluna].dropna()
        print(f"{cidade:>14}  {rotulo:>24}  {s.mean():6.3f}  "
              f"{s.std():7.3f}  {s.quantile(.95):6.3f}")
    print()


curitiba       células= 4398  NaN por eixo -> IDS: 43  Esgoto inadequado: 43  Renda: 43  Taxa de analfabetismo: 43
porto_alegre   células= 5065  NaN por eixo -> IDS: 336  Esgoto inadequado: 336  Renda: 336  Taxa de analfabetismo: 336

        cidade                      eixo   média   desvio     p95
      curitiba            IDS (composto)   0.405    0.201   0.756
      curitiba      Esgoto inadequado   0.398    0.231   0.836
      curitiba         Renda (invertida)   0.573    0.229   0.903
      curitiba     Taxa de analfabetismo   0.243    0.213   0.656

  porto_alegre            IDS (composto)   0.475    0.198   0.764
  porto_alegre      Esgoto inadequado   0.375    0.221   0.777
  porto_alegre         Renda (invertida)   0.690    0.235   0.998
  porto_alegre     Taxa de analfabetismo   0.360    0.304   1.000



## 4. Um painel por cidade

Cada cidade ganha sua própria figura, com os quatro indicadores lado a lado. Duas figuras separadas porque as manchas têm proporções diferentes e porque a leitura de cada cidade interessa em separado.

Dentro de cada figura, todos os mapas usam a mesma escala de 0 a 1 e a mesma paleta, então o tom de uma célula significa a mesma coisa nos quatro indicadores.


In [39]:
from matplotlib.cm import ScalarMappable

for cidade, nome in CIDADES.items():
    malha = malhas[cidade]

    fig, eixos = plt.subplots(
        1, 4, figsize=(26, 9),
        gridspec_kw={"wspace": 0.01},
    )

    for coluna, (nome_coluna, rotulo) in enumerate(EIXOS):
        ax = eixos[coluna]
        malha.plot(
            column=nome_coluna,
            cmap=CMAP,
            norm=NORM,
            ax=ax,
            edgecolor="none",
            missing_kwds={"color": "#e0e0e0"},
        )
        ax.set_aspect("equal")
        ax.set_axis_off()
        ax.set_title(rotulo, fontsize=19, pad=12, fontweight="bold")

    # Barra de cores única para os quatro mapas da cidade.
    cax = fig.add_axes([0.32, 0.045, 0.36, 0.014])
    barra = fig.colorbar(ScalarMappable(norm=NORM, cmap=CMAP), cax=cax,
                         orientation="horizontal", ticks=np.arange(0, 1.01, 0.2))
    barra.set_label("Índice normalizado (1 = maior vulnerabilidade)",
                    fontsize=17, labelpad=14)
    barra.ax.tick_params(labelsize=14)

    fig.suptitle(f"Desigualdade socioespacial por eixo — {nome}",
                 fontsize=25, y=0.99)
    destino = OUTPUT_DIR / f"ids_by_axis_{cidade}.png"
    fig.savefig(destino, dpi=250, bbox_inches="tight")
    plt.close(fig)
    print("salvo:", destino.name)


salvo: ids_by_axis_curitiba.png
salvo: ids_by_axis_porto_alegre.png
